# Simple model specification workflow

This notebook shows the config-to-spec project workflow:

1. Create or load a grid
2. Build a `SimpleModelConfig`
3. Translate it with `simple_model_spec(...)`
4. Execute a named run through `Project`
5. Reopen the run for result analysis

The example uses a tiny hand-built Voronoi grid so the setup stays focused on composing and running the model.


In [ ]:
from pathlib import Path

import numpy as np
import myflopy as mf

In [ ]:
# A tiny 2-cell Voronoi grid with clockwise polygons.
verts = np.array(
    [
        [0.0, 0.0],
        [1.0, 0.0],
        [1.0, 1.0],
        [0.0, 1.0],
        [2.0, 0.0],
        [2.0, 1.0],
    ],
    dtype=float,
)
iverts = [[0, 3, 2, 1], [1, 2, 5, 4]]
xcyc = np.array([[0.5, 0.5], [1.5, 0.5]], dtype=float)

vor = mf.VoronoiGridPlus(verts=verts, iverts=iverts, xcyc=xcyc)
vor.gdf_vorPolys

In [ ]:
# Keep the model name at 16 characters or fewer because MF6 enforces that limit.
project_root = Path.cwd().resolve().parents[1] / "artifacts" / "simple_model_workflow"

config = mf.SimpleModelConfig(
    vor=vor,
    name="smpl_demo",
    nper=1,
    nlay=1,
    grid_type="disv",
    top=[10.0, 9.0],
    bottom=[[0.0, 0.0]],
    initial_heads=[10.0, 9.0],
    k=[1.0, 1.0],
    save_specific_discharge=False,
    boundary_mode="chd",
    boundary_cells=[0, 1],
    boundary_head=[10.0, 9.0],
    sto_steady={0: True},
    sto_transient={},
)
config


In [ ]:
spec = mf.simple_model_spec(config)
project = mf.Project(project_root, name="simple-model-demo")
run = project.run("baseline", spec, overwrite=True)
run.success


In [ ]:
flow = run.simulation.get_model("smpl_demo")
heads = flow.output.head().get_data(kstpkper=(0, 0)).squeeze()
heads


In [ ]:
completed = mf.load_mf6_run(run.workspace)
completed.all_heads[["elev", "geometry"]]


## Where to go from here

- Replace the hand-built `VoronoiGridPlus` with a grid built from `TriangleGrid`
- Swap `boundary_mode="chd"` for `"drain"` or `None`
- Add `rch_dict` for recharge-driven examples
- Replace individual package specs with `ModelSpec.with_package(...)` to define scenarios
- Reopen completed runs with `load_mf6_run(...)` for heads, budgets, plots, and post-processing
